# E3 · Primer ensayo de desarrollo en Colab

**CNN-LSTM causal con atención penalizada por discordancia y saturación radial**, frente al control `M00` sin ambas modificaciones.

1. En **Entorno de ejecución → Cambiar tipo de entorno**, selecciona una GPU. Una A100 es suficiente; el código comprueba la GPU realmente asignada.
2. Ejecuta las celdas en orden. La instalación inicial puede tardar varios minutos y no reinicia el kernel. Acepta el montaje de tu Drive cuando Colab lo solicite.
3. La celda **Entrenar o reanudar** inicia el ensayo: una semilla, hasta 40 épocas por modelo y **20 minutos en total para ambos**, incluyendo validación y guardado. Las épocas se alternan.
4. Si la sesión se interrumpe, vuelve a abrir este mismo notebook y ejecuta las celdas en orden con la misma carpeta de Drive.

Los resultados son de desarrollo y la validación guía la parada temprana. No prueban superioridad frente al Top 10, no verifican la novedad y no sustituyen la evaluación final de E3. Este notebook no abre los controladores de E2.

In [ ]:
from pathlib import Path
import hashlib
import json
import os
import subprocess
import sys
import urllib.request

REPOSITORY = "jassan-bit/DeepLearning_202630"
SOURCE_COMMIT = "b705787bdde3d93d57eae2deb726280c9720027c"
DEVICE = "cuda"
LOCAL_WORKSPACE = Path("/content/entregable3_development_v1")
# Conserva esta ruta al reconectar: identifica una única corrida y su saldo.
RUN_DIR = Path("/content/drive/MyDrive/DeepLearning_202630/E3/desarrollo_v1")
SOURCE_DIR = LOCAL_WORKSPACE / ("source-" + SOURCE_COMMIT)
print("Código fijado:", SOURCE_COMMIT)
print("Resultados persistentes:", RUN_DIR)

## Preparar el código y el entorno aislado

Se descargan únicamente los archivos necesarios desde un commit inmutable. El entorno usa Python 3.12.12, PyTorch 2.8.0 con CUDA 12.6 y NumPy 2.3.5. Su instalación queda en el disco temporal de Colab; los resultados quedan en Drive.

In [ ]:
SOURCE_FILES = [
    "src/__init__.py", "src/original_model.py", "src/e3_development.py",
    "scripts/e3_run_development.py", "experiments/e3_desarrollo_v1/plan.json",
    "notebooks/preparar_e3_desarrollo_runtime.py",
    "requirements-e3-development-gpu.lock.txt",
]

def download_pinned(relative_path, commit, destination, expected_sha256=None):
    url = f"https://raw.githubusercontent.com/{REPOSITORY}/{commit}/{relative_path}"
    with urllib.request.urlopen(url, timeout=120) as response:
        payload = response.read()
    digest = hashlib.sha256(payload).hexdigest()
    if expected_sha256 is not None and digest != expected_sha256:
        raise RuntimeError("SHA-256 distinto para " + relative_path)
    destination = Path(destination)
    destination.parent.mkdir(parents=True, exist_ok=True)
    if destination.exists():
        if destination.read_bytes() != payload:
            raise RuntimeError("Archivo local modificado; se conserva: " + str(destination))
    else:
        temporary = destination.with_suffix(destination.suffix + ".download")
        temporary.write_bytes(payload)
        os.replace(temporary, destination)
    return digest

source_hashes = {name: download_pinned(name, SOURCE_COMMIT, SOURCE_DIR / name)
                 for name in SOURCE_FILES}
import importlib.util
helper_path = SOURCE_DIR / "notebooks/preparar_e3_desarrollo_runtime.py"
spec = importlib.util.spec_from_file_location("e3_isolated_runtime", helper_path)
helper = importlib.util.module_from_spec(spec)
spec.loader.exec_module(helper)
GPU_PYTHON, RUN_ENV, GPU_RUNTIME = helper.preparar_entorno(
    LOCAL_WORKSPACE / "runtime", device=DEVICE)

## Conectar Drive y verificar los datos

El archivo auditado está fijado por SHA-256. El entrenador sólo lee los tres arreglos de entrenamiento originales. Recalcula el preprocesamiento con **532 ejemplos de ajuste**, purga siete días y reserva **116 ejemplos para validación interna**. No lee los arreglos de calibración ni de evaluación.

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

PLAN_PATH = SOURCE_DIR / "experiments/e3_desarrollo_v1/plan.json"
PLAN = json.loads(PLAN_PATH.read_text())
data_source = PLAN["data_source"]
DATA_PATH = RUN_DIR.parent / "datos_auditados_v1" / "current_cohort_arrays.npz"
DATA_SHA256 = data_source["sha256"]
# Reutilizar los bytes verificados de Drive evita otra descarga al reconectar.
if DATA_PATH.exists():
    if hashlib.sha256(DATA_PATH.read_bytes()).hexdigest() != DATA_SHA256:
        raise RuntimeError("El archivo de datos de Drive no coincide con la auditoría; se conserva.")
else:
    download_pinned(data_source["path"], data_source["commit"], DATA_PATH, DATA_SHA256)
print("Datos verificados:", DATA_SHA256)
print("Fuente de los datos:", data_source["commit"])

# Comprobar las operaciones usadas para conservar los puntos de reanudación.
# El bloqueo se verifica entre dos procesos de esta misma VM.
import fcntl
import tempfile
RUN_DIR.parent.mkdir(parents=True, exist_ok=True)
with tempfile.TemporaryDirectory(prefix=".e3_storage_probe_", dir=RUN_DIR.parent) as probe:
    probe = Path(probe)
    first, second = probe / "first", probe / "second"
    with first.open("wb") as stream:
        stream.write(b"e3-storage-probe")
        stream.flush()
        os.fsync(stream.fileno())
    os.replace(first, second)
    assert second.read_bytes() == b"e3-storage-probe"
    with (probe / "lock").open("a+") as lock_stream:
        fcntl.flock(lock_stream, fcntl.LOCK_EX | fcntl.LOCK_NB)
        child = chr(10).join(["import fcntl,sys", "with open(sys.argv[1], 'a+') as f:",
                            " try:", "  fcntl.flock(f,fcntl.LOCK_EX|fcntl.LOCK_NB)",
                            " except BlockingIOError:", "  sys.exit(42)", " sys.exit(1)"])
        outcome = subprocess.run([sys.executable, "-c", child, str(probe / "lock")],
                                 capture_output=True, text=True)
        if outcome.returncode != 42:
            raise RuntimeError("Drive no superó la comprobación del bloqueo entre procesos; conserva esta salida para revisarla.")
print("Guardado y bloqueo local comprobados. Usa una sola copia de este notebook por corrida.")

## Comprobación previa: preparar sin entrenar

Esta celda crea el registro de la corrida y sus pesos iniciales. Si ya existe la corrida, conserva sus archivos para reanudarla. La celda siguiente volverá a comprobar el código, los datos, el plan y el tipo de dispositivo antes de continuar.

In [ ]:
CLI = SOURCE_DIR / "scripts/e3_run_development.py"
COMMON_ARGS = ["--npz", str(DATA_PATH), "--expected-sha256", DATA_SHA256,
               "--run-dir", str(RUN_DIR), "--device", DEVICE]
if not RUN_DIR.exists():
    subprocess.run([str(GPU_PYTHON), str(CLI), "preflight", *COMMON_ARGS,
                    "--plan", str(PLAN_PATH)], cwd=SOURCE_DIR, env=RUN_ENV, check=True)
else:
    required = ["manifest.json", "plan.json", "state.json", "ledger.json", "data_admission.json"]
    if any(not (RUN_DIR / name).is_file() for name in required):
        raise RuntimeError("Corrida parcial: se conserva para inspección. No se reinicia ni se borra.")
    print("Corrida existente conservada. Se usará la reanudación con su saldo registrado.")

admission = json.loads((RUN_DIR / "data_admission.json").read_text())
assert admission["npz_sha256"] == DATA_SHA256
assert admission["n_fit"] == data_source["expected_fit_count"]
assert admission["n_validation"] == data_source["expected_validation_count"]
assert admission["n_purged_or_unused"] == data_source["expected_purged_count"]
assert admission["keys_read"] == ["training_raw", "training_y_original", "training_dates"]
ledger = json.loads((RUN_DIR / "ledger.json").read_text())
print(json.dumps({"ajuste": admission["n_fit"], "validacion_interna": admission["n_validation"],
                  "purgados": admission["n_purged_or_unused"],
                  "limite_segundos": ledger["limit_seconds"],
                  "segundos_consumidos": ledger["charged_seconds"],
                  "saldo_segundos": max(0, ledger["limit_seconds"] - ledger["charged_seconds"])},
                 ensure_ascii=False, indent=2))

## Entrenar o reanudar

Al ejecutar esta celda se inicia el entrenamiento. El tiempo registrado incluye los guardados frecuentes en Drive. El registro conserva lo consumido al reanudar y se detiene al terminar ambos modelos, activarse la parada temprana o agotarse el saldo. No cambies el plan ni la carpeta para recuperar tiempo. Cada época informa su avance.

Para detenerlo, usa el botón de parada de esta celda. Si Colab se cierra abruptamente, la recuperación puede cargar conservadoramente el intervalo hasta la siguiente conexión y agotar el saldo. El código conserva los puntos de reanudación y verifica su integridad.

In [ ]:
state = json.loads((RUN_DIR / "state.json").read_text())
command = "run" if state["status"] == "initialized" else "resume"
args = [str(GPU_PYTHON), "-u", str(CLI), command, *COMMON_ARGS]
log_path = RUN_DIR / "colab_console.log"
# El proceso se interrumpe al detener esta celda; no se deja un controlador de fondo.
process = subprocess.Popen(args, cwd=SOURCE_DIR, env=RUN_ENV,
                           stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                           text=True, bufsize=1, start_new_session=True)
try:
    with log_path.open("a", encoding="utf-8") as log:
        for line in process.stdout:
            print(line, end="", flush=True)
            log.write(line)
            log.flush()
    return_code = process.wait()
except KeyboardInterrupt:
    print("Interrupción solicitada. Conserva la misma carpeta para reanudar.")
    raise
finally:
    # También cerrar el hijo si falla la escritura del log de Drive.
    if process.poll() is None:
        import signal
        for stop_signal, timeout in ((signal.SIGINT, 30), (signal.SIGTERM, 10), (signal.SIGKILL, 5)):
            try:
                os.killpg(process.pid, stop_signal)
            except ProcessLookupError:
                break
            try:
                process.wait(timeout=timeout)
                break
            except subprocess.TimeoutExpired:
                continue
if return_code:
    raise RuntimeError(f"El ensayo se detuvo con código {return_code}. Revisa la salida y {log_path}.")

## Examinar el resultado de desarrollo

RMSE y MAE están en las unidades originales de σ7. Cada modelo se resume en su mejor época según validación; esa selección hace que estas métricas sean exploratorias. La curva de pérdida de ajuste es la media de pérdidas por lote durante las actualizaciones, no una evaluación del checkpoint final. El control `M00` tiene un parámetro menos y retira dos componentes a la vez.

In [ ]:
from html import escape
from IPython.display import HTML, display

summary_path = RUN_DIR / "summary.json"
if not summary_path.is_file():
    print("Aún no hay resumen: revisa la salida del entrenamiento y los archivos de la corrida.")
else:
    summary = json.loads(summary_path.read_text())
    print("Estado:", summary["status"])
    print(f"Tiempo registrado: {summary['charged_seconds']:.1f} / {summary['limit_seconds']:.0f} s")
    rows = []
    for name, result in summary["models"].items():
        values = [name, result["parameters"], result["completed_epochs"], result["best_epoch"],
                  result["best_validation_rmse"], result["best_validation_mae"]]
        rows.append("<tr>" + "".join("<td>" + escape(str(value)) + "</td>" for value in values) + "</tr>")
    display(HTML("<table><thead><tr><th>Modelo</th><th>Parámetros</th><th>Épocas completas</th>"
                 "<th>Mejor época</th><th>RMSE validación interna</th><th>MAE validación interna</th>"
                 "</tr></thead><tbody>" + "".join(rows) + "</tbody></table>"))
    print("Guarda estos archivos:", RUN_DIR)
    print("summary.json, plan.json, manifest.json, ledger.json, history.jsonl, data_admission.json")
    print("Cada modelo: curvas, predicciones de validación, mejores pesos y punto de reanudación.")

In [ ]:
import csv
try:
    import plotly.graph_objects as go
except ImportError:
    print("Plotly no está disponible en el kernel; las curvas están en los CSV de Drive.")
else:
    figure = go.Figure()
    for name in ("full", "M00"):
        curves_path = RUN_DIR / name / "curves.csv"
        if curves_path.is_file():
            with curves_path.open() as stream:
                rows = list(csv.DictReader(stream))
            figure.add_trace(go.Scatter(x=[int(row["epoch"]) for row in rows],
                                       y=[float(row["validation_RMSE_original"]) for row in rows],
                                       mode="lines+markers", name=name))
    figure.update_layout(title="Validación interna de desarrollo · una semilla",
                         xaxis_title="Época", yaxis_title="RMSE de σ7 (unidades originales)",
                         template="plotly_white")
    figure.show()

## Siguiente decisión

Revisar las curvas, el coste, los errores y la comparación contra `M00`. Una mejora aquí permite continuar estudiando el mecanismo; una ausencia de mejora requiere analizar la causa y registrar la decisión. No se eligen semillas ni conjuntos a posteriori para producir significancia.

Antes de una campaña final siguen pendientes la revisión de novedad, la comprensión del código por el autor, las ablaciones individuales, el control de capacidad, el análisis de potencia y el protocolo pre-registrado. La declaración de asistencia de IA debe incluir este trabajo.